# SimLab demo (slew + detumble)

Lightweight walkthrough for **Dynamics-calc**. It runs the existing SimLab CLI for `--scenario slew` and `--scenario detumble`, then points at the committed recruiter figures (slew PNG/GIF from M1, Monte Carlo PNGs from PR #10).

This notebook does **not** rewrite the plant, controllers, or estimators. Optional RKMK4 is a **library** switch on `step_rigid_body(..., method="rkmk4")` — it is **not** a SimLab CLI flag. Closed-loop runs stay on default RK4.

**CWD:** repo root (`Dynamics-calc/`). Headless matplotlib is already `Agg` inside `attitude_sim.plots`.

## Install (once)

```bash
python -m venv .venv
source .venv/bin/activate
pip install -e ".[dev]"
```

Jupyter is optional (`pip install jupyter` / VS Code / GitHub preview). The cells below only need the package CLI.

## Slew CLI

Recruiter-length run (writes `outputs/slew_summary.png` and `outputs/slew_attitude.gif`):

```bash
python -m attitude_sim --scenario slew
```

Regenerate the committed copies in `docs/figures/`:

```bash
python -m attitude_sim --scenario slew --out-dir docs/figures
```

The next cell is a **short** Agg PNG smoke (`--no-gif`, same policy as CI). It does not overwrite `docs/figures/`.

In [ ]:
import subprocess
from pathlib import Path

out = Path("outputs") / "demo"
out.mkdir(parents=True, exist_ok=True)
subprocess.check_call(
    [
        "python",
        "-m",
        "attitude_sim",
        "--scenario",
        "slew",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "slew_summary.png")

**Committed rest-to-rest slew summary** (`docs/figures/slew_summary.png`). Default stack `pid` + `mekf`, 75°, 40 s.

![Rest-to-rest slew summary (pid + mekf)](../docs/figures/slew_summary.png)

**Committed attitude GIF** (`docs/figures/slew_attitude.gif`). CI does **not** smoke GIFs.

![Rest-to-rest slew attitude animation](../docs/figures/slew_attitude.gif)

## Detumble CLI

Named SimLab scenario (not part of the Monte Carlo harness):

```bash
python -m attitude_sim --scenario detumble
python -m attitude_sim --scenario detumble --out-dir outputs --no-gif
```

In [ ]:
subprocess.check_call(
    [
        "python",
        "-m",
        "attitude_sim",
        "--scenario",
        "detumble",
        "--t-final",
        "0.5",
        "--no-gif",
        "--out-dir",
        str(out),
    ]
)
print("wrote", out / "detumble_summary.png")

## Monte Carlo figures (PR #10)

`python -m attitude_sim.monte_carlo` is **slew-only**. Committed summary PNGs:

- `docs/figures/mc_final_att_error_hist.png` — final geodesic attitude-error histogram
- `docs/figures/mc_settle_vs_noise.png` — settle-time proxy vs log-uniform sensor-noise scale

![Monte Carlo final attitude-error histogram](../docs/figures/mc_final_att_error_hist.png)

![Monte Carlo settle time vs sensor-noise scale](../docs/figures/mc_settle_vs_noise.png)

Regenerate the committed copies (overwrites `docs/figures/mc_*.png`):

```bash
python -m attitude_sim.monte_carlo --n 40 --seed 0 --estimator mekf \
    --t-final 40 --inertia-frac 0.05 \
    --plot --out-dir docs/figures \
    --json outputs/mc_slew.json --csv outputs/mc_slew.csv
```

Re-plot from a previous JSON (no new trials):

```bash
python -m attitude_sim.monte_carlo --from-json outputs/mc_slew.json \
    --plot --out-dir docs/figures
```

CI covers the Agg PNG helpers (`tests/test_monte_carlo.py`) plus a check that these files exist (`tests/test_docs_figures.py`). GIF smoke is skipped.